# Refuel & Finetune — Resume from latest_384.pth (IMG_SIZE=384, BATCH_SIZE=16)

Bu notebok `latest_384.pth` checkpoint'inden devam ederek 30 epoch daha eğitir. Kriteler:
- IMG_SIZE = 384, BATCH_SIZE = 16
- Omurga LR = 5e-6, ArcFace head LR = 5e-5
- Scheduler: `ReduceLROnPlateau(mode='min', factor=0.5, patience=3)` — validation loss durağanlaşınca LR yarıya düşsün
- Checkpoint `model_state_dict`, `metric_fc_state_dict` ve (varsa) `optimizer_state_dict` yüklenecek; yoksa optimizer yeniden başlatılacak ve devam edilecek.

Kullanım: Colab ortamında çalıştırın; Drive'ınıza repo/ckpt konumunu mount edin ve `CHECKPOINT_PATH`'i güncelleyin.

In [ ]:
# Mount Google Drive and set paths
from google.colab import drive
drive.mount('/content/drive')

# Update these to match your Drive repo layout
REPO_DIR = '/content/drive/MyDrive/kaplumbaga_tanima'
CHECKPOINT_PATH = '/content/drive/MyDrive/kaplumbaga_tanima/src/models/latest_384.pth'  # <-- update if needed
OUTPUT_DIR = '/content/drive/MyDrive/kaplumbaga_tanima/checkpoints_refuel_384'
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Install dependencies (timing: quick). Colab already has torch; we ensure timm and other libs.
!pip install -q timm tqdm

In [ ]:
# Imports and basic helpers
import sys, time, json, copy, math, os
import numpy as np
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
from tqdm import tqdm
import timm

# Add repo to path to import local modules if present
if REPO_DIR not in sys.path:
    sys.path.append(REPO_DIR)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
# Training params (CRITICAL settings as requested)
IMG_SIZE = 384
BATCH_SIZE = 16
BACKBONE_LR = 5e-6
HEAD_LR = 5e-5
EPOCHS = 30
S = 30.0  # ArcFace scale if used in head
M = 0.5   # ArcFace margin if used in head

TRAIN_DIR = os.path.join(REPO_DIR, 'datasets', 'train')
VAL_DIR = os.path.join(REPO_DIR, 'datasets', 'test')

print('TRAIN_DIR', TRAIN_DIR)
print('VAL_DIR', VAL_DIR)
print('CHECKPOINT_PATH', CHECKPOINT_PATH)

In [ ]:
# Attempt to import project model components; fallback to inline definitions if not available
try:
    from src.models.turtle_model import TurtleModel
    from src.models.arcface import ArcFace
    print('Imported TurtleModel & ArcFace from repo')
    USE_REPO_IMPL = True
except Exception as e:
    print('Could not import local modules, using inline definitions. Error:', e)
    USE_REPO_IMPL = False

if not USE_REPO_IMPL:
    class TurtleModel(nn.Module):
        def __init__(self, emb_size=512, pretrained=True):
            super().__init__()
            self.backbone = timm.create_model('convnext_tiny.fb_in22k_ft_in1k', pretrained=pretrained, num_classes=0, global_pool='')
            in_ch = self.backbone.num_features if hasattr(self.backbone, 'num_features') else 768
            self.head = nn.Sequential(
                nn.AdaptiveAvgPool2d(1),
                nn.Flatten(),
                nn.Linear(in_ch, emb_size),
                nn.BatchNorm1d(emb_size)
            )
        def forward(self, x):
            x = self.backbone(x) if isinstance(self.backbone, nn.Module) and hasattr(self.backbone, 'num_features') else x
            emb = self.head(x)
            emb = nn.functional.normalize(emb, p=2, dim=1)
            return emb

    class ArcFace(nn.Module):
        def __init__(self, in_features=512, out_features=1000, s=30.0, m=0.5):
            super().__init__()
            self.weight = nn.Parameter(torch.Tensor(out_features, in_features))
            nn.init.xavier_uniform_(self.weight)
            self.s = s
            self.m = m
        def forward(self, x, labels=None):
            # If labels provided, compute logits with margin; otherwise cosine logits
            cosine = nn.functional.linear(nn.functional.normalize(x), nn.functional.normalize(self.weight))
            if labels is None:
                return cosine * self.s
            # margin operations (simple implementation)
            theta = torch.acos(torch.clamp(cosine, -1.0+1e-7, 1.0-1e-7))
            phi = torch.cos(theta + self.m)
            one_hot = torch.zeros_like(cosine)
            one_hot.scatter_(1, labels.view(-1,1), 1.0)
            logits = (one_hot * phi) + ((1.0 - one_hot) * cosine)
            logits *= self.s
            return logits

    print('Inline model and ArcFace ready')

In [ ]:
# Data transforms and loaders
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.1,0.1,0.1,0.1),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])
val_transform = transforms.Compose([
    transforms.Resize(int(IMG_SIZE*1.15)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

train_ds = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
val_ds = datasets.ImageFolder(VAL_DIR, transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

num_classes = len(train_ds.classes)
print('Classes:', num_classes)

In [ ]:
# Build model and metric head
EMB_SIZE = 512
model = TurtleModel(emb_size=EMB_SIZE).to(device)
metric_fc = ArcFace(in_features=EMB_SIZE, out_features=num_classes, s=S, m=M).to(device)

# Optimizer with two param groups (backbone, head)
backbone_params = [p for n, p in model.named_parameters() if p.requires_grad]
head_params = [p for n, p in metric_fc.named_parameters() if p.requires_grad]
optimizer = optim.AdamW([{'params': backbone_params, 'lr': BACKBONE_LR}, {'params': head_params, 'lr': HEAD_LR}], weight_decay=1e-6)

# Criterion (standard CE)
criterion = nn.CrossEntropyLoss()

# Scheduler: ReduceLROnPlateau (use validation loss)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, verbose=True)

# Helper to save checkpoint
def save_ckpt(path, epoch, model, metric_fc, optimizer, best_val_loss, is_best=False):
    payload = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'metric_fc_state_dict': metric_fc.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'best_val_loss': best_val_loss
    }
    torch.save(payload, path)
    if is_best:
        best_path = os.path.join(os.path.dirname(path), 'best_384.pth')
        torch.save(payload, best_path)

# Load checkpoint (resume)
start_epoch = 0
best_val_loss = float('inf')
if os.path.exists(CHECKPOINT_PATH):
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
    try:
        model.load_state_dict(ckpt['model_state_dict'])
        metric_fc.load_state_dict(ckpt['metric_fc_state_dict'])
        print('Loaded model and metric_fc from checkpoint')
    except Exception as e:
        print('Warning: failed to load model state exactly:', e)
    if 'optimizer_state_dict' in ckpt:
        try:
            optimizer.load_state_dict(ckpt['optimizer_state_dict'])
            print('Loaded optimizer state')
        except Exception as e:
            print('Warning: failed to load optimizer state:', e)
    start_epoch = ckpt.get('epoch', 0)
    best_val_loss = ckpt.get('best_val_loss', float('inf'))
    print(f'Resuming from epoch {start_epoch}, best_val_loss={best_val_loss}')
else:
    print('Checkpoint not found at', CHECKPOINT_PATH, '; training from scratch with requested LRs')

In [ ]:
# Training and validation loops
def validate(model, metric_fc, dataloader, device):
    model.eval(); metric_fc.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for imgs, labels in tqdm(dataloader, desc='Val', leave=False):
            imgs = imgs.to(device)
            labels = labels.to(device)
            emb = model(imgs)
            logits = metric_fc(emb, labels) if hasattr(metric_fc, 'forward') else metric_fc(emb)
            loss = criterion(logits, labels)
            running_loss += loss.item() * imgs.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += imgs.size(0)
    avg_loss = running_loss / total if total>0 else float('inf')
    acc = correct / total if total>0 else 0.0
    return avg_loss, acc

def train_one_epoch(epoch, model, metric_fc, loader, optimizer, device):
    model.train(); metric_fc.train()
    running_loss = 0.0
    correct = 0
    total = 0
    pbar = tqdm(loader, desc=f'Train Epoch {epoch}', leave=False)
    for imgs, labels in pbar:
        imgs = imgs.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        emb = model(imgs)
        logits = metric_fc(emb, labels) if hasattr(metric_fc, 'forward') else metric_fc(emb)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += imgs.size(0)
        pbar.set_postfix({'loss': running_loss/total, 'acc': f'{100.*correct/total:.2f}%'})
    return running_loss / total, correct / total

# Main loop: run EPOCHS epochs after start_epoch
global_start = start_epoch
for e in range(start_epoch + 1, start_epoch + 1 + EPOCHS):
    t0 = time.time()
    train_loss, train_acc = train_one_epoch(e, model, metric_fc, train_loader, optimizer, device)
    val_loss, val_acc = validate(model, metric_fc, val_loader, device)
    scheduler.step(val_loss)  # ReduceLROnPlateau uses validation loss
    is_best = val_loss < best_val_loss
    if is_best:
        best_val_loss = val_loss
    ckpt_path = os.path.join(OUTPUT_DIR, f'refuel_epoch_{e}.pth')
    save_ckpt(ckpt_path, e, model, metric_fc, optimizer, best_val_loss, is_best=is_best)
    t1 = time.time()
    print(f'Epoch {e} finished. Train loss {train_loss:.4f}, Train acc {100.*train_acc:.2f}%, Val loss {val_loss:.4f}, Val acc {100.*val_acc:.2f}% — Time {(t1-t0)/60:.2f}m')

print('Training completed — checkpoints saved to', OUTPUT_DIR)

## Notes
- If `optimizer_state_dict` is not present in `latest_384.pth`, the notebook will continue training with a freshly-initialized optimizer using the requested LRs. This is intentional to avoid mismatch errors.
- `ReduceLROnPlateau` requires a reliable validation loss; ensure `VAL_DIR` has proper labels.